In [1]:
#imports

import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [21]:
def getData(lat=52.52, long=13.41, start="2025-10-26", end="2025-11-09", features=["weather_code"]):
    # Setup the Open-Meteo API client with cache and retry on error
    cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
    retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
    openmeteo = openmeteo_requests.Client(session = retry_session)

    # Make sure all required weather variables are listed here
    # The order of variables in hourly or daily is important to assign them correctly below
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
	"latitude": lat,
	"longitude": long,
	"start_date": start,
	"end_date": end,
    "timezone": "America/Denver",
	"daily": features,
	"temperature_unit": "fahrenheit",
	"wind_speed_unit": "mph",
	"precipitation_unit": "inch",
    }
    responses = openmeteo.weather_api(url, params=params)
    return responses

In [22]:
def dataToDataframe(response, features=["weather_code"]):
    daily = response.Daily()

    daily_data = {"date": pd.date_range(
	    start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
	    end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
	    freq = pd.Timedelta(seconds = daily.Interval()),
	    inclusive = "left"
    )}

    for idx, feat in enumerate(features):
        daily_data[feat] = daily.Variables(idx).ValuesAsNumpy()

    df = pd.DataFrame(data=daily_data)
    return df


In [23]:
#Set Parameters
features = ["weather_code", "temperature_2m_mean", "sunrise", "sunset", "snowfall_sum", "precipitation_sum", "rain_sum", "wind_speed_10m_max"]
lat = 43.373508
long = -112.135382
startDate = "2015-01-01"
endDate = "2025-10-31"

In [24]:
resp = getData(lat=lat, long=long, start=startDate, end=endDate, features=features)
df = dataToDataframe(resp[0], features=features)

In [25]:
df.head()

,date,weather_code,temperature_2m_mean,sunrise,sunset,snowfall_sum,precipitation_sum,rain_sum,wind_speed_10m_max
0,2015-01-01 07:00:00+00:00,2.0,2.064200,0,0,0.000000,0.0,0.0,4.112674
1,2015-01-02 07:00:00+00:00,2.0,4.651699,0,0,0.000000,0.0,0.0,3.355500
2,2015-01-03 07:00:00+00:00,3.0,13.539200,0,0,0.000000,0.0,0.0,3.809474
3,2015-01-04 07:00:00+00:00,71.0,21.871698,0,0,0.137795,0.0,0.0,9.059159
4,2015-01-05 07:00:00+00:00,71.0,34.370449,0,0,0.055118,0.0,0.0,19.544006


In [ ]:
#Not it became clear after performed the api call that sunrise/sunse where in fact all zeros shown below. 
#Thus they can be dropped from the dataframe

print(df['sunrise'].value_counts())
print(df['sunset'].value_counts())
df.drop(labels=['sunrise','sunset'], axis=1, inplace=True)

sunrise
0    3957
Name: count, dtype: int64
sunset
0    3957
Name: count, dtype: int64


In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3957 entries, 0 to 3956
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   date                 3957 non-null   datetime64[ns, UTC]
 1   weather_code         3957 non-null   float32            
 2   temperature_2m_mean  3957 non-null   float32            
 3   snowfall_sum         3957 non-null   float32            
 4   precipitation_sum    3957 non-null   float32            
 5   rain_sum             3957 non-null   float32            
 6   wind_speed_10m_max   3957 non-null   float32            
dtypes: datetime64[ns, UTC](1), float32(6)
memory usage: 123.8 KB
